In [1]:
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np
import time
import random
from gwgraphs import *

In [2]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [3]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("BZR_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def construct_networkx_graph(edges, graph_indicators, graph_labels, node_labels,
                           node_attributes):
    graphs = {}
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr) in enumerate(zip(graph_indicators, node_labels,
                                            node_attributes),
                                        start=1):
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr

    return graphs


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix

In [4]:
class GraphNeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=3):
        super(GraphNeuralNetwork, self).__init__()
        
        self.num_layers = num_layers
        self.convs = nn.ModuleList()
        
        # First layer
        self.convs.append(GCNConv(input_dim, hidden_dim))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            
        # Last layer
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        
        # Classifier
        self.classifier = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x, edge_index, batch):
        # Apply GCN layers
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = self.dropout(x)
                
        # Global pooling to get graph-level representation
        x = global_mean_pool(x, batch)
        
        # Final classification
        x = self.classifier(x)
        
        return x
    

def train_model(model, train_loader, optimizer, critierion, device):
    """Function to train the model for one epoch."""
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        out = model(batch.x, batch.edge_index, batch.batch)
        
        # Convert labels from {-1, 1} to {0,1} for cross-entropy loss
        labels = (batch.y + 1)//2
        
        loss = criterion(out, labels.view(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(train_loader)


def evaluate_model(model, test_loader, device):
    """Function to evaluate the model."""
    model.eval()
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            
            # Get predictions
            pred = torch.argmax(out, dim=1)
            # Convert back to {-1,1} labels
            pred = pred * 2 - 1
            
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(batch.y.cpu().numpy())
            
        return np.array(predictions), np.array(true_labels)
    
    
def calculate_metrics(y_true, y_pred):
    """Calculate classification metrics"""
    # Overall accuracy
    overall_acc = accuracy_score(y_true, y_pred)

    # By-class accuracy
    class_1_mask = y_true == 1
    class_neg1_mask = y_true == -1

    class_1_acc = accuracy_score(
        y_true[class_1_mask],
        y_pred[class_1_mask]) if np.sum(class_1_mask) > 0 else 0
    class_neg1_acc = accuracy_score(
        y_true[class_neg1_mask],
        y_pred[class_neg1_mask]) if np.sum(class_neg1_mask) > 0 else 0

    # F1 score
    f1 = f1_score(y_true, y_pred, average='weighted')

    return overall_acc, class_1_acc, class_neg1_acc, f1


def networkx_to_pyg(nx_graph, label, num_features):
    """Convert NetworkX graph to PyTorch Geometric Data object"""
    num_nodes = len(nx_graph.nodes)

    # Extract node features from the graph's node attributes
    node_features = []
    node_list = sorted(nx_graph.nodes())  # Ensure consistent ordering
    
    for node in node_list:
        if 'attributes' in nx_graph.nodes[node]:
            # Use the actual node attributes from the BZR dataset
            node_features.append(nx_graph.nodes[node]['attributes'])
        else:
            # Fallback to zero features if attributes are missing
            node_features.append([0.0] * num_features)
    
    # Convert to tensor
    x = torch.tensor(node_features, dtype=torch.float32)

    # Convert edges to tensor format - need to map node IDs to indices
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    edge_list = list(nx_graph.edges())
    
    if len(edge_list) == 0:
        # Handle graphs with no edges
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        # Map node IDs to indices for edge_index
        mapped_edges = [(node_to_idx[u], node_to_idx[v]) for u, v in edge_list]
        edge_index = torch.tensor(mapped_edges, dtype=torch.long).t().contiguous()

    # Convert label to tensor
    y = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, y=y)

In [5]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data()
graphs = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )
total_avg, class_averages = compute_averages(graphs)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

NameError: name 'compute_averages' is not defined

In [6]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]

In [7]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model parameters
input_dim = 3  # Number of node features; BZR dataset has 3D node attributes
hidden_dim = 64  # defines the number of neurons in the GCN layers. Essentially, controls the wedith of the neural network
# larger hidden_dim means more capacity but more paramters
output_dim = 2  # Binary classification, so set to 2
num_epochs = 50
learning_rate = 0.01
batch_size = 32

# Convert NetworkX graphs to PyTorch Geometric Data objects
full_graph_data = []
for graph_id, nx_graph in graphs.items():
    label = nx_graph.graph['label']
    pyg_data = networkx_to_pyg(nx_graph, label, input_dim)
    full_graph_data.append(pyg_data)

# Extract labels for reference
labels = [data.y.item() for data in full_graph_data]
labels = np.array(labels)
print("BZR dataset processed successfully.")

# 5-fold cross validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

all_results = []
fold_times = []

print("\nStarting 5-fold cross-validation...")
total_start_time = time.time()

for fold, (train_idx, test_idx) in enumerate(kfold.split(full_graph_data)):
    print(f"\n--- Fold {fold + 1}/5 ---")
    fold_start_time = time.time()

    # Split data
    train_graphs = [full_graph_data[i] for i in train_idx]
    test_graphs = [full_graph_data[i] for i in test_idx]

    # Create data loaders
    train_loader = DataLoader(train_graphs,
                              batch_size=batch_size,
                              shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

    # Initialize model
    model = GraphNeuralNetwork(input_dim, hidden_dim, output_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    for epoch in range(num_epochs):
        train_loss = train_model(model, train_loader, optimizer, criterion,
                                 device)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_loss:.4f}")

            # Evaluation
            y_pred, y_true = evaluate_model(model, test_loader, device)

            # Calculate metrics
            overall_acc, class_1_acc, class_neg1_acc, f1 = calculate_metrics(
                y_true, y_pred)

            fold_time = time.time() - fold_start_time
            fold_times.append(fold_time)

            results = {
                'fold': fold + 1,
                'overall_accuracy': overall_acc,
                'class_1_accuracy': class_1_acc,
                'class_-1_accuracy': class_neg1_acc,
                'f1_score': f1,
                'fold_time': fold_time
            }

            all_results.append(results)

            print(f"Results for Fold {fold + 1}:")
            print(f"  Overall Accuracy: {overall_acc:.4f}")
            print(f"  Class +1 Accuracy: {class_1_acc:.4f}")
            print(f"  Class -1 Accuracy: {class_neg1_acc:.4f}")
            print(f"  F1 Score: {f1:.4f}")
            print(f"  Fold Time: {fold_time:.2f} seconds")

            total_time = time.time() - total_start_time

            # Calculate average metrics across all folds
            avg_overall_acc = np.mean([r['overall_accuracy'] for r in all_results])
            avg_class_1_acc = np.mean([r['class_1_accuracy'] for r in all_results])
            avg_class_neg1_acc = np.mean([r['class_-1_accuracy'] for r in all_results])
            avg_f1 = np.mean([r['f1_score'] for r in all_results])

            std_overall_acc = np.std([r['overall_accuracy'] for r in all_results])
            std_class_1_acc = np.std([r['class_1_accuracy'] for r in all_results])
            std_class_neg1_acc = np.std([r['class_-1_accuracy'] for r in all_results])
            std_f1 = np.std([r['f1_score'] for r in all_results])

            print("\n" + "=" * 50)
            print("FINAL RESULTS - 5-FOLD CROSS-VALIDATION")
            print("=" * 50)
            print(
                f"Average Overall Accuracy: {avg_overall_acc:.4f} ± {std_overall_acc:.4f}"
            )
            print(
                f"Average Class +1 Accuracy: {avg_class_1_acc:.4f} ± {std_class_1_acc:.4f}"
            )
            print(
                f"Average Class -1 Accuracy: {avg_class_neg1_acc:.4f} ± {std_class_neg1_acc:.4f}"
            )
            print(f"Average F1 Score: {avg_f1:.4f} ± {std_f1:.4f}")
            print(f"\nTotal Training and Evaluation Time: {total_time:.2f} seconds")
            print(f"Average Time per Fold: {np.mean(fold_times):.2f} seconds")

Using device: cpu
BZR dataset processed successfully.

Starting 5-fold cross-validation...

--- Fold 1/5 ---


C:\Users\Kaitlyn\anaconda3\Lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 10/50, Loss: 0.4505
Results for Fold 1:
  Overall Accuracy: 0.8272
  Class +1 Accuracy: 0.0000
  Class -1 Accuracy: 1.0000
  F1 Score: 0.7489
  Fold Time: 1.08 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8272 ± 0.0000
Average Class +1 Accuracy: 0.0000 ± 0.0000
Average Class -1 Accuracy: 1.0000 ± 0.0000
Average F1 Score: 0.7489 ± 0.0000

Total Training and Evaluation Time: 1.09 seconds
Average Time per Fold: 1.08 seconds
Epoch 20/50, Loss: 0.4275
Results for Fold 1:
  Overall Accuracy: 0.7901
  Class +1 Accuracy: 0.1429
  Class -1 Accuracy: 0.9254
  F1 Score: 0.7604
  Fold Time: 2.25 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8086 ± 0.0185
Average Class +1 Accuracy: 0.0714 ± 0.0714
Average Class -1 Accuracy: 0.9627 ± 0.0373
Average F1 Score: 0.7546 ± 0.0057

Total Training and Evaluation Time: 2.25 seconds
Average Time per Fold: 1.67 seconds
Epoch 30/50, Loss: 0.4151
Results for Fold 1:
  Overall Accuracy: 0.8395
 

C:\Users\Kaitlyn\anaconda3\Lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 10/50, Loss: 0.4336
Results for Fold 2:
  Overall Accuracy: 0.7654
  Class +1 Accuracy: 0.0000
  Class -1 Accuracy: 1.0000
  F1 Score: 0.6637
  Fold Time: 1.16 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8066 ± 0.0243
Average Class +1 Accuracy: 0.1786 ± 0.1637
Average Class -1 Accuracy: 0.9502 ± 0.0461
Average F1 Score: 0.7625 ± 0.0492

Total Training and Evaluation Time: 7.38 seconds
Average Time per Fold: 3.22 seconds
Epoch 20/50, Loss: 0.4533
Results for Fold 2:
  Overall Accuracy: 0.7654
  Class +1 Accuracy: 0.0000
  Class -1 Accuracy: 1.0000
  F1 Score: 0.6637
  Fold Time: 2.61 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8007 ± 0.0267
Average Class +1 Accuracy: 0.1531 ± 0.1639
Average Class -1 Accuracy: 0.9574 ± 0.0461
Average F1 Score: 0.7484 ± 0.0572

Total Training and Evaluation Time: 8.83 seconds
Average Time per Fold: 3.14 seconds
Epoch 30/50, Loss: 0.4367
Results for Fold 2:
  Overall Accuracy: 0.7778
 

...

Epoch 10/50, Loss: 0.4291
Results for Fold 3:
  Overall Accuracy: 0.7654
  Class +1 Accuracy: 0.0500
  Class -1 Accuracy: 1.0000
  F1 Score: 0.6751
  Fold Time: 1.38 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8025 ± 0.0298
Average Class +1 Accuracy: 0.1785 ± 0.1734
Average Class -1 Accuracy: 0.9685 ± 0.0416
Average F1 Score: 0.7488 ± 0.0608

Total Training and Evaluation Time: 14.25 seconds
Average Time per Fold: 3.56 seconds
Epoch 20/50, Loss: 0.4599
Results for Fold 3:
  Overall Accuracy: 0.7778
  Class +1 Accuracy: 0.1500
  Class -1 Accuracy: 0.9836
  F1 Score: 0.7166
  Fold Time: 2.54 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8004 ± 0.0293
Average Class +1 Accuracy: 0.1761 ± 0.1662
Average Class -1 Accuracy: 0.9697 ± 0.0400
Average F1 Score: 0.7461 ± 0.0589

Total Training and Evaluation Time: 15.41 seconds
Average Time per Fold: 3.47 seconds
Epoch 30/50, Loss: 0.4011
Results for Fold 3:
  Overall Accuracy: 0.7901

C:\Users\Kaitlyn\anaconda3\Lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 10/50, Loss: 0.5258
Results for Fold 4:
  Overall Accuracy: 0.8148
  Class +1 Accuracy: 0.0625
  Class -1 Accuracy: 1.0000
  F1 Score: 0.7427
  Fold Time: 1.32 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8009 ± 0.0258
Average Class +1 Accuracy: 0.2016 ± 0.1630
Average Class -1 Accuracy: 0.9670 ± 0.0371
Average F1 Score: 0.7511 ± 0.0522

Total Training and Evaluation Time: 20.82 seconds
Average Time per Fold: 3.67 seconds
Epoch 20/50, Loss: 0.4553
Results for Fold 4:
  Overall Accuracy: 0.8148
  Class +1 Accuracy: 0.1875
  Class -1 Accuracy: 0.9692
  F1 Score: 0.7735
  Fold Time: 3.10 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8017 ± 0.0252
Average Class +1 Accuracy: 0.2008 ± 0.1581
Average Class -1 Accuracy: 0.9672 ± 0.0360
Average F1 Score: 0.7524 ± 0.0509

Total Training and Evaluation Time: 22.59 seconds
Average Time per Fold: 3.64 seconds
Epoch 30/50, Loss: 0.3907
Results for Fold 4:
  Overall Accuracy: 0.8395

C:\Users\Kaitlyn\anaconda3\Lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 10/50, Loss: 0.4720
Results for Fold 5:
  Overall Accuracy: 0.7901
  Class +1 Accuracy: 0.0000
  Class -1 Accuracy: 1.0000
  F1 Score: 0.6975
  Fold Time: 1.46 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8095 ± 0.0313
Average Class +1 Accuracy: 0.2102 ± 0.1641
Average Class -1 Accuracy: 0.9720 ± 0.0343
Average F1 Score: 0.7609 ± 0.0562

Total Training and Evaluation Time: 28.65 seconds
Average Time per Fold: 3.87 seconds
Epoch 20/50, Loss: 0.4364
Results for Fold 5:
  Overall Accuracy: 0.8148
  Class +1 Accuracy: 0.2353
  Class -1 Accuracy: 0.9688
  F1 Score: 0.7779
  Fold Time: 3.26 seconds

FINAL RESULTS - 5-FOLD CROSS-VALIDATION
Average Overall Accuracy: 0.8098 ± 0.0306
Average Class +1 Accuracy: 0.2113 ± 0.1604
Average Class -1 Accuracy: 0.9718 ± 0.0336
Average F1 Score: 0.7617 ± 0.0551

Total Training and Evaluation Time: 30.45 seconds
Average Time per Fold: 3.84 seconds
Epoch 30/50, Loss: 0.4154
Results for Fold 5:
  Overall Accuracy: 0.7901